# 특허 기술이전 수요예측 — Colab GPU 실행 (최종)

맥북 잠자기/뚜껑 걱정 없이 **무료 T4 GPU**로 돌립니다. **이제 중단돼도 같은 셀을 다시 실행하면 죽은 시드부터 이어서 진행**됩니다(자동 resume).

## ⚠️ 가장 먼저 (필수)

상단 메뉴 **런타임 ▸ 런타임 유형 변경 ▸ T4 GPU ▸ 저장** 으로 GPU를 켜세요. 그다음 셀을 **순서대로** 실행하세요. (끊기면 GPU 설정 후 **셀 1**부터 다시 — 셀 1은 몇 번 돌려도 안전)

## 셀 1 — 셋업 (GPU확인 + 코드 + 라이브러리 + 데이터, 한 번에)
재활용/끊김에 강하게 통합. 이 한 셀만 다시 돌리면 환경이 복구됩니다. **`git pull`로 최신 코드(resume · full_pool 포함)를 받습니다.**

In [ ]:
import torch, os
assert torch.cuda.is_available(), '❌ GPU 없음! 런타임▸런타임 유형 변경▸T4 GPU 설정 후 다시 실행하세요.'
if not os.path.exists('/content/repo'):
    !git clone https://github.com/joshlee614/Patent-Citation-Graph-Based-Technology-Transfer-Demand-Prediction-.git /content/repo
%cd /content/repo
!git pull -q
!pip install -q torch_geometric statsmodels
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/kipris_data'   # ← 본인 데이터 폴더 경로 확인
EMB = os.path.join(DATA_DIR, 'patent_embeddings.pt')
import torch_geometric
print('✅ GPU:', torch.cuda.get_device_name(0), '| DATA OK:', os.path.exists(EMB))
!git log --oneline -1

## 셀 2 — 빠른 검증 (seeds 3, ~25분)

파이프라인이 정상 도는지 먼저 확인(메인 결과 아님). 결과는 **드라이브에 직접 저장**. 끊기면 이 셀을 그대로 다시 실행하면 이어서 진행됩니다.

In [ ]:
!python run_ipm_experiment.py --mode full --seeds 3 --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" --demand_sample 200 \
  --split temporal \
  --artifact_dir /content/drive/MyDrive/kipris_data/ipm_check_s3

## 셀 3 — 최종 temporal 10-seed (~80분, **메인 결과**)

현실 프로토콜(시간분할 + 동일-IPC 하드네거티브). `--nneg_sweep`(후보군 크기 민감도) + `--full_pool`(비샘플 전체-풀 §12, 샘플드-메트릭 방어)까지 포함.
**끊기면 이 셀 그대로 다시 실행 → 죽은 시드부터 이어서.**

In [ ]:
!python run_ipm_experiment.py --mode full --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" --demand_sample 200 \
  --split temporal --nneg_sweep --full_pool \
  --artifact_dir /content/drive/MyDrive/kipris_data/ipm_results_final

## 셀 4 — 최종 random 10-seed (~80분, **RQ1 대조**)

랜덤 분할(누수 있는 비교군). temporal과의 대조가 RQ1. `--full_pool`로 대조도 비샘플 풀에서 확인.
**끊기면 이 셀 그대로 다시 실행.** (temporal과 폴더가 달라 절대 안 섞임)

In [ ]:
!python run_ipm_experiment.py --mode full --device cuda \
  --data_dir "{DATA_DIR}" --emb_path "{EMB}" --demand_sample 200 \
  --split random --full_pool \
  --artifact_dir /content/drive/MyDrive/kipris_data/ipm_results_random

## 셀 5 — 결과 확인 (temporal + random, 절대경로라 cwd 무관)

In [ ]:
import os
for name, path in [('TEMPORAL (main)', '/content/drive/MyDrive/kipris_data/ipm_results_final/run_ipm_results.md'),
                   ('RANDOM (RQ1)',    '/content/drive/MyDrive/kipris_data/ipm_results_random/run_ipm_results.md')]:
    print('='*80); print(name, '->', path); print('='*80)
    print(open(path).read() if os.path.exists(path) else '(아직 없음 — 해당 셀 완료 후 생성됨)')
    print()

---
### 끊김 / 재개 (중요 — 이제 안전)
- **죽어도 같은 셀을 그대로 다시 실행**하면 `<artifact_dir>/_resume_checkpoint.pt`에서 **죽은 시드부터 이어서** 진행합니다(완료 시드 건너뜀, Demand-BFS 등 무거운 전처리도 복원).
- 끊기면 순서: GPU 설정 확인 → **셀 1**(환경 복구) → 죽었던 **그 셀** 다시.
- 같은 `--artifact_dir`로 돌려야 이어집니다. 완전히 처음부터 새로 하려면 셀의 명령에 `--fresh_start` 추가.
- 체크포인트는 매 시드 드라이브에 저장돼서 후반엔 한 번 쓰는 데 수십 초 걸릴 수 있음(정상).
- **탭 활성 유지 + 맥 안 자게**(맥 터미널 `caffeinate -d`). GPU 계속 안 잡히면 무료 quota 소진(~12~24h 후 재시도 또는 다른 계정).